In [2]:
import pandas as pd
import glob

In [ ]:
def carregar_dados(path_pattern):

    files = glob.glob(path_pattern)

    df = pd.concat(
        [pd.read_parquet(f) for f in files],
        ignore_index=True
    )
    return df

In [ ]:
import numpy as np
import json
import os

STATS_PATH = "/data/features/stats/stats_referencia.json"
MIN_DIAS_PARA_CONGELAR = 30

def calcular_ou_carregar_stats(df):
    dias_disponiveis = df['date'].nunique()

    if os.path.exists(STATS_PATH):
        with open(STATS_PATH) as f:
            return json.load(f)

    if dias_disponiveis < MIN_DIAS_PARA_CONGELAR:
        print(f"[AVISO] Apenas {dias_disponiveis} dias — stats provisórios - régua ainda instável")
        congelar = False
    else:
        congelar = True

    media_log = np.log1p(
        df['mentions'] * 0.5 +
        df['articles'] * 0.3 +
        df['sources']  * 0.2
    )
    stats = {
        "p5_media":  media_log.quantile(0.05),
        "p95_media": media_log.quantile(0.95),
        "p5_wti":    None,  # preenchido depois do primeiro calcular_wti completo
        "p95_wti":   None
    }

    if congelar:
        with open(STATS_PATH, "w") as f:
            json.dump(stats, f)
        print(f"[INFO] Stats congelados com {dias_disponiveis} dias de histórico")

    return stats


In [5]:


def calcular_wti(df):
    df = df.copy()

    stats = calcular_ou_carregar_stats(df)

    # Goldstein
    df['goldstein_norm'] = (10 - df['goldstein']) / 20

    # Tone 
    df['tone_norm'] = df['tone'].clip(upper=0).abs() / 100

    # Media weight 
    media_log = np.log1p(
        df['mentions'] * 0.5 +
        df['articles'] * 0.3 +
        df['sources']  * 0.2
    )
    p5_m  = stats["p5_media"]
    p95_m = stats["p95_media"]

    df['media_weight'] = (
        (media_log - p5_m) / (p95_m - p5_m)
    ).clip(0, 1)

    # WTI raw 
    df['wti_raw'] = (
        df['goldstein_norm'] * 0.6 +
        df['tone_norm']      * 0.2 +
        df['media_weight']   * 0.2
    )

    # WTI final 
    if stats["p5_wti"] is None:
        # Ainda sem referência congelada — usa o próprio df
        w5  = df['wti_raw'].quantile(0.05)
        w95 = df['wti_raw'].quantile(0.95)
    else:
        w5  = stats["p5_wti"]
        w95 = stats["p95_wti"]

    df['wti'] = (
        (df['wti_raw'] - w5) / (w95 - w5)
    ).clip(0, 1)

    # Congela wti stats junto se ainda não foi feito
    if stats["p5_wti"] is None and os.path.exists(STATS_PATH):
        stats["p5_wti"]  = df['wti_raw'].quantile(0.05)
        stats["p95_wti"] = df['wti_raw'].quantile(0.95)
        with open(STATS_PATH, "w") as f:
            json.dump(stats, f)

    return df

In [ ]:
df = carregar_dados("/data/processed/*.parquet",
                    "/data/features/gold.parquet")
df = df.groupby(["date", "actor1_country"]).agg({
    "goldstein": "mean",
    "tone": "mean",
    "mentions": "sum",
    "sources": "sum",
    "articles": "sum"
}).reset_index()
df = df.sort_values(["actor1_country", "date"])
df = calcular_wti(df)
df.to_parquet("/data/features/gold.parquet", index=False)

In [12]:
print(df.head())

           date actor1_country  goldstein      tone  mentions  sources  \
948  2025-01-01            ABW   1.700000  5.882353        14        2   
1143 2025-01-02            ABW   1.080000  0.247898        34        5   
1341 2025-01-03            ABW  -2.000000  0.000000        10        1   
1543 2025-01-04            ABW   0.933333  4.505623        15        3   
1740 2025-01-05            ABW   1.400000  0.558831        16        2   

      articles  goldstein_norm  tone_norm  media_weight   wti_raw       wti  
948         14        0.415000        0.0      0.193670  0.287734  0.303348  
1143        34        0.446000        0.0      0.329463  0.333493  0.434923  
1341        10        0.600000        0.0      0.142858  0.388572  0.593298  
1543        15        0.453333        0.0      0.206009  0.313202  0.376578  
1740        16        0.430000        0.0      0.212985  0.300597  0.340334  
